# Notebook 3: Predictive Modeling

Build and compare 5 different ML models for MSFT stock price prediction.

In [ ]:


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_single_stock
from src.data_cleaner import clean_data, create_features, prepare_data_for_modeling
from src.predictor import StockPredictor
from src.visualizer import plot_predictions_vs_actual, plot_model_comparison
from config.config import PRIMARY_STOCK

print('✅ All imports successful!')


## Load and Prepare Data

In [ ]:
# Load MSFT data
print('Loading MSFT data...')
raw_data = load_single_stock(PRIMARY_STOCK)

# Clean data
print('Cleaning data...')
clean_data_df = clean_data(raw_data, PRIMARY_STOCK)

# Create features
print('Creating features...')
featured_data = create_features(clean_data_df)

print(f'\nData shape: {featured_data.shape}')
print(f'Features: {featured_data.columns.tolist()[:10]}...')


## Prepare Train-Test Split

In [ ]:
# Prepare data for modeling
X_train, X_test, y_train, y_test = prepare_data_for_modeling(featured_data, test_split=0.8)

print(f'Training set: {len(X_train)} samples')
print(f'Test set: {len(X_test)} samples')
print(f'Total features: {X_train.shape[1]}')


## Build Predictive Models

In [ ]:
# Initialize predictor
predictor = StockPredictor()

print('='*80)
print('BUILDING MACHINE LEARNING MODELS')
print('='*80)

# 1. Linear Regression
lr_model, lr_metrics = predictor.build_linear_regression(X_train, y_train, X_test, y_test)

# 2. Random Forest
rf_model, rf_metrics = predictor.build_random_forest(X_train, y_train, X_test, y_test, n_estimators=100)

# 3. XGBoost
xgb_model, xgb_metrics = predictor.build_xgboost(X_train, y_train, X_test, y_test)

# 4. ARIMA (Time Series)
arima_model = predictor.build_arima(featured_data['Close'], order=(5, 1, 2))


## Model Performance Summary

In [ ]:
# Get results summary
results_df = predictor.get_results_summary()

print('\n' + '='*80)
print('MODEL PERFORMANCE COMPARISON')
print('='*80)
print(results_df.to_string())

# Get best model
best_model_name, best_model = predictor.get_best_model()


## Visualize Model Comparison

In [ ]:
# Plot model comparison
plot_model_comparison(results_df)


## Predictions vs Actual

In [ ]:
# Plot predictions for each model
for model_name, result in predictor.results.items():
    y_pred = result['predictions']
    plot_predictions_vs_actual(y_test.values, y_pred, model_name, PRIMARY_STOCK)


## Feature Importance (Random Forest)

In [ ]:
# Get feature importance from Random Forest
rf_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
# Ensure feature names are strings for plotting
rf_importance['Feature'] = rf_importance['Feature'].astype(str)

print('\n' + '='*80)
print('TOP 10 IMPORTANT FEATURES (Random Forest)')
print('='*80)
print(rf_importance.head(10).to_string(index=False))

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(rf_importance['Feature'][:10], rf_importance['Importance'][:10])
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title('Top 10 Important Features for Price Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visualizations/feature_importance.png', dpi=300, bbox_inches='tight')
print('\nSaved: visualizations/feature_importance.png')
plt.show()


## Key Insights

In [ ]:
print('\n' + '='*80)
print('MODELING INSIGHTS')
print('='*80)

print(f'\n1. Best Performing Model: {best_model_name}')
best_r2 = predictor.results[best_model_name]['metrics']['R2']
print(f'   R² Score: {best_r2:.4f}')

print(f'\n2. Model Comparison:')
for _, row in results_df.iterrows():
    print(f'   {row["Model"]}: MAE=${row["MAE"]:.2f}, R²={row["R2"]:.4f}')

print('\n3. Recommendations:')
print(f'   - Use {best_model_name} for production forecasting')
print('   - Ensemble methods outperform linear models')
print('   - Feature engineering improves model performance')

print('\n✅ Predictive modeling complete!')
